# Reverse Engineer Triton Fused Attention Kernel

Here I am trying to break-down all the minute details of a triton fused attention kernel. For best understanding please keep the fused attention kernel code open in one tab and read through this. I have tried to explain all the small details line by line.

## Introduction

Attention is a mechanism to identify the maximum probability for the next word in a sentence. This is a main component of LLM on which the LLMs learn to write proper sentences. So based on context length, LLM tries to learn and predict the next word. However when we are training LLMs on GPUs the main problem which comes into account is we cannot train it for longer context length as we lack memory, we don't use enough FLOPS of a GPUs, we cannot do distributed computing. Now with the ascent of Triton and other GPU optimization ways. We can now break the data into small BLOCKs and process them.

Important thing to note is we make matrix based operation faster by breaking it into smaller block and processing blocks using parallel threads. We wrap them into group of threads. This helps fetching of data faster as parallel threads do it. However 1 triton program is executed in a serial fashion as if it is single threaded. However since we execute triton programs in a grid which can be max of 3 dimensions we execute all those programs in parallel. This makes life easier for a triton programmer. As he doesn't need to worry about parallelism, only need to find the best ways to write a triton kernel.

However there are few ways in which we make use of the LRU cache as reusing data cached across multiple triton programs. This helps us reduce the time of fetching the data block. This is called overlapping pattern of Triton. This is the most commonly used pattern in Triton.

## Notations used for reverse engineering

Few specification regarding it,

1. We use tl.program() which can have 3 values given in the grid. So if my gird is [10,20,4] then I can have 3 triton program ids where each is accessed by
tl.program(0), tl.program(1), tl.program(2) and the total number of programs are 10x20x4 = 800 programs. So to simulate using torch I need to use 3 nested for loops which will simulate the offsets.

2. We have used some places torch.randn to identify the the whole method and use the offset's shape to create the tensors

3. We have used normal division instead of tl.cdiv

4. I have removed all the GPU specfic code to better understanding


## Attention

Lets talk about attention as a beginner. Attention is nothing but a MAP of key and value where a query is given to check if the key is present on not. This is the simplist way to explain attention. Now lets add probability to it. Suppose a key can have multiple values and each value has a probability. So based on the query we get the list of probablity of values and we generate a sample out of this words list and the value which has the highest probabilty appears as a sample output.

Lets talk about the mathematics around it,

$$
Attention = softmax(qk^T)V
$$

Here $q$ is the query, $k$ is the key and $V$ is the value

Since we are writing a fused attention kernel. We need to define the forward and the backward parts of it. Which is used to identify the loss and based on it train the parameters effectively.

Since we are dealing with matrixes, here is the dimensions of the input

[Z, H, N_CTX, HEAD_DIM] is the basic dimension of query, key and value.

Where,
* Z/BATCH : Gives us the no of items in a batch
* H : Gives us the no. of heads in a attention
* N_CTX : No of tokens in a context(context length)
* HEAD_DIM : No of dimensions per token.

If you see how multi-head attention works we divide the HEAD_DIM by the H. So suppose if my head dimension is 512 and no. of heads is 8 then each head will have 512/8 = 64 head dimensions. However we don't break the HEAD_DIM here as it remains constant per HEAD.

### Masking

Masking is important part of Attention mechanism. Here we create mask for context length 10 as follows.
$$
\begin{pmatrix}
True & False & False & False & False & False & False & False & False & False \\
True & True  & False & False & False & False & False & False & False & False \\
True & True  & True  & False & False & False & False & False & False & False \\
True & True  & True  & True  & False & False & False & False & False & False \\
True & True  & True  & True  & True  & False & False & False & False & False \\
True & True  & True  & True  & True  & True  & False & False & False & False \\
True & True  & True  & True  & True  & True  & True  & False & False & False \\
True & True  & True  & True  & True  & True  & True  & True  & False & False \\
True & True  & True  & True  & True  & True  & True  & True  & True  & False \\
True & True  & True  & True  & True  & True  & True  & True  & True  & True \\
\end{pmatrix}
$$

Now if we use triton blocks to create this suppose of size 5x2 then it becomes
Mask for diagonal elements are
$$
\,
$$
$$
M00 =  
\begin{pmatrix}
True & False \\
True & True  \\
True & True  \\
True & True  \\
True & True  \\
\end{pmatrix}
, \, M01 =
\begin{pmatrix}
False & False \\
False & False  \\
True & False  \\
True & True  \\
True & True  \\
\end{pmatrix}
, \, M02 =
\begin{pmatrix}
False & False \\
False & False  \\
False & False  \\
False & False  \\
True & False  \\
\end{pmatrix}
$$
$$
\,
$$
$$
M10 =  
\begin{pmatrix}
True & True \\
True & True  \\
True & True  \\
True & True  \\
True & True  \\
\end{pmatrix}
, \, M11 =  
\begin{pmatrix}
True & True \\
True & True  \\
True & True  \\
True & True  \\
True & True  \\
\end{pmatrix}
$$
$$
\,
$$
$$
M12 =  
\begin{pmatrix}
True & True \\
True & True  \\
True & True  \\
True & True  \\
True & True  \\
\end{pmatrix}
, \, M13 =  
\begin{pmatrix}
False & False \\
True & False  \\
True & True  \\
True & True  \\
True & True  \\
\end{pmatrix}
, \, M14 =  
\begin{pmatrix}
False & False \\
False & False  \\
False & False  \\
True & False  \\
True & True  \\
\end{pmatrix}
$$
$$
\,
$$
And rest of the values are not even taken into consideration

In [ ]:
import torch
import os

DEVICE = "cude" if torch.cuda.is_available() else 'cpu'

## DEMO of how offsets are masked

offs_m = torch.arange(10, 20)
offs_n = 2+torch.arange(0, 5)
mask = (offs_m[:, None] >= offs_n[None, :])
print(f"Offset of n ({offs_n.shape}): {offs_n}")
print(f"Offset of m ({offs_m.shape}) : {offs_m}")
print(f"Mask ({mask.shape}) : {mask}")


Offset of n (torch.Size([5])): tensor([2, 3, 4, 5, 6])
Offset of m (torch.Size([10])) : tensor([10, 11, 12, 13, 14, 15, 16, 17, 18, 19])
Mask (torch.Size([10, 5])) : tensor([[True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True]])


## Forward

This is nothing but the way to execute the forward kernel. We have 2 stages in which we calculate the $qk$ and apply the mask. Here I am talking the causal part of thing where we train attention with only past data and we mask the future data. This predicts the next token/word and used in calulation of the loss. This mode is called causal mode (causal = True).

Here we use stage 1 for non-masking values and stage 2 for only mask values. We dont even calculate the $qk$ part which will be not used at all. If we see the above masks M00, M01, M02, M13, M14 are the stage 2 values where mask is applied where as M10, M11, M12 are the stage 1 values where mask is not applied.

Now we can start on the details of the code used. Here we use
```
v = v.permute(0, 1, 3, 2).contiguous()
v = v.permute(0, 1, 3, 2)
```
This makes the value contigous and transposes it as we need to mutliply it with the softmax of $qk$ which makes the mulitplcation faster.

Next thing to see is while so compare this code with actual code provided as a triton sample. You see that the grid : (16, 32, 1). So total programs that will run are 16x32x1 = 512 in parallel. This is based on
```
grid = (q.shape[2]// BLOCK_M, q.shape[0] * q.shape[1], 1)
```
So as you can see based on this grid since q.shape[0] is BATCH dimension and q.shape[1] is no. of HEADs. So the 2 value in the grid represents the total no of times N_CTX per batch per head. Which is further divided into BLOCK_M. So 1 tl program represents N_CTX per BLOCK_M per HEAD per BATCH.
```
off_z = off_hz // H
off_h = off_hz % H
```
When you see this kind of notation, this means that we are dividing the off_hz which is coming from q.shape[0] * q.shape[1]. which is divided by H so H become total no of columns and off_z gives us the row_index and off_h gives us the column index. This is a common pattern used for converting list of values into a matrix in triton.
```
H = 8
VAL = 100
for i in range(VAL):
    print(f"Row : {i//H} : Columns : {i%H}")
```  
Use this to identify what i mean.

Now lets talk about the y-dimension which we see that the BATCH * H * N_CTX.
```
tl.make_tensor_descriptor(desc_q, shape=[y_dim, HEAD_DIM], strides=[HEAD_DIM, 1],block_shape=[BLOCK_M, HEAD_DIM])
```
This is used to create the tensor descriptor for triton. This is nothing but a way to create a triton tensor. Where all the other dimensions are taken as y_dim and HEAD_DIM is taken as columns. Now we need to iterate with an offset along y direction.
```
offset_y = off_z * (N_CTX * H) + off_h * N_CTX
```
Which is dont by offset_y. For offset of q we are executing on causal=true all the values of k and v so we multiply the grid[0] which is start_m with BLOCK_M so we get BLOCK_M chunks with the offset. This way before calling _att_fwd_inner we have already made blocks of q in y_dimension.

We also form offsets fo offs_m to store the maximum of all the values in a column which helps us in softmax calculation.

Inside _att_fwd_inner function we use 2 variable lo and hi. Here we determine the masking & non-masking parts
```
if STAGE == 1:
        lo, hi = 0, start_m * BLOCK_M
elif STAGE == 2:
    lo, hi = start_m * BLOCK_M, (start_m + 1) * BLOCK_M
```

Here you can see if y_dim = 16 and BLOCK_M = 4 then we get 4 blocks along x and y directions. Based on the above code lo and hi for stage 1 covers all non-masking blocks and stage 2 covers all the masking blocks. Below is the visual representation of the matrix formed by $qk$
$$
\begin{bmatrix}
Mask &  &  &  \\
No-Mask & Mask  &  &  \\
No-Mask & No-Mask  & Mask  &   \\
No-Mask & No-Mask  & No-Mask  & Mask   \\
\end{bmatrix}
$$

Now we run the from lo to hi with an increment of BLOCK_N. However before that we set offsets of k and v along y direction using (for fp8 machines we tend to tranpose it for faster operation which is not much of a benefit in fp16 and above machines)
```
offsetk_y = offset_y + lo
offsetv_y = offset_y + lo
```
Now we can talk about the loop(lo to hi with increment of BLOCK_N). Here we use mask creation with the below code.
```
mask = offs_m[:, None] >= (start_n + offs_n[None, :])
```
### Softmax in triton

Softmax in triton is applied in a interesting way. Lets go through the details of it. Since we only have access to BLOCK of data not the whole tensor. We need to find a way to apply softmax in a most effective way. To do this we find the current max and store it in m_ij and keep on updating in m_i where keeps the max value from other blocks.

If you see the intialization before passing to the inner function is this

```
m_i = tl.zeros([BLOCK_M], dtype=tl.float32) - float("inf")
```
and using it as
```
m_ij = tl.maximum(m_i, tl.max(qk, 1) * qk_scale)
```
Here we set it to negative infinity keep on updating the value based on the iteration it is running on. Important thing is we scale all the values by multiplying it with sm_scale (1/log(2)) as we are using 2 as base for softmax rather than exponential(e) as as base. This is shown by below code
```
p = tl.math.exp2(qk)
```
Since we are using running max rather than actual max so we have to do correction if the max comes in the next iteration. This is done by alpha
```
alpha = tl.math.exp2(m_i - m_ij)
```
We do accumulation correction using the alpha
```
acc = acc * alpha[:, None]
```
and then we accumulate the result using
```
acc = tl.dot(p, v, acc)
```
The same correction is applied to the denominator as well which is l_ij (running sum) and l_i which is the current sum of values. Then we iterate over the k and v by BLOCK_N

Finally we do finish the softmax as
```
acc = acc / l_i[:, None]
```

### Function

In case of Forward function we have used tensor descriptors which just provides the block just by giving the offset to it. Which is simple and cleaner. Since BLOCK_M is 64 and BLOCK_N is 32 so the offset_m and offset_n is defined the same. Some important values to note is
1. When start_m = 0, off_hz = 0 : q start from 0 with BLOCK_M and qo_offset_y = 0 and it is fed to the tensor descriptor to get q

2. When start_m = 0, off_hz = 1 : q start from 1024 with BLOCK_M as the N_CTX is moved and we process the N_CTX per HEAD per BATCH and qo_offset_y = 1024 and it is fed to the tensor descriptor to get q

3. When start_m = 1, off_hz = 0 : q start from 64 with BLOCK_M as we process the next set of tokens of the same N_CTX and qo_offset_y = 64 and it is fed to the tensor descriptor to get q

4. When start_m = 1, off_hz = 1 : q start from 1024+64=1088 with BLOCK_M as we process the next set of tokens of the next N_CTX and qo_offset_y = 1024+64 = 1088 and it is fed to the tensor descriptor to get q


In [29]:

def _attn_fwd_inner(acc, l_i, m_i, q,  #
                    desc_k, desc_v,  #
                    offset_y, dtype, start_m, qk_scale,  #
                    BLOCK_M, HEAD_DIM, BLOCK_N,  #
                    STAGE, offs_m, offs_n,  #
                    N_CTX, warp_specialize):
    # range of values handled by this stage
    print(STAGE)
    if STAGE == 1:
        lo, hi = 0, start_m * BLOCK_M
    elif STAGE == 2:
        lo, hi = start_m * BLOCK_M, (start_m + 1) * BLOCK_M
        # lo = tl.multiple_of(lo, BLOCK_M)
    # causal = False
    else:
        lo, hi = 0, N_CTX
    offsetk_y = offset_y + lo

    #dtype == tl.float8e5
    if dtype == torch.float8_e5m2:
        offsetv_y = offset_y * HEAD_DIM + lo
    else:
        offsetv_y = offset_y + lo

    # loop over k, v and update accumulator
    # TODO what is warp_specialize in tl.range ?
    # for start_n in tl.range(lo, hi, BLOCK_N, warp_specialize=warp_specialize):
    for start_n in torch.arange(lo, hi, BLOCK_N):
        # start_n = tl.multiple_of(start_n, BLOCK_N)
        # -- compute qk ----
        print(f"start_n : {start_n} : offset of k [{offsetk_y},0]")
        # k = desc_k.load([offsetk_y, 0]).T
        # qk = tl.dot(q, k)
        if STAGE == 2:
            mask = offs_m[:, None] >= (start_n + offs_n[None, :])
            print(f"mask is used {mask}")
            # qk = qk * qk_scale + tl.where(mask, 0, -1.0e6)
            # m_ij = tl.maximum(m_i, tl.max(qk, 1))
            # qk -= m_ij[:, None]
        else:
            print("no mask used")
            # m_ij = tl.maximum(m_i, tl.max(qk, 1) * qk_scale)
            # qk = qk * qk_scale - m_ij[:, None]
        # p = tl.math.exp2(qk)
        # -- compute correction factor
        # alpha = tl.math.exp2(m_i - m_ij)
        # l_ij = tl.sum(p, 1)
        # -- update output accumulator --
        # acc = acc * alpha[:, None]
        # prepare p and v for the dot
        print(f"Offset of v [0, {offsetv_y}]")
        # if dtype == tl.float8e5:
        #     v = desc_v.load([0, offsetv_y]).T
        # else:
        #     v = desc_v.load([offsetv_y, 0])
        # p = p.to(dtype)
        # note that this non transposed v for FP8 is only supported on Blackwell
        # acc = tl.dot(p, v, acc)
        # update m_i and l_i
        # place this at the end of the loop to reduce register pressure
    #     l_i = l_i * alpha + l_ij
    #     m_i = m_ij
        offsetk_y += BLOCK_N
        offsetv_y += BLOCK_N
    # return acc, l_i, m_i
    print("----inner-----")

def _attn_fwd(sm_scale, M, Z, H, desc_q, desc_k, desc_v, desc_o,
              HEAD_DIM,  #
              BLOCK_M,  #
              BLOCK_N,  #
              FP8_OUTPUT,  #
              STAGE,  #
              warp_specialize, N_CTX):
  dtype = torch.float16
  assert BLOCK_N <= HEAD_DIM
  # start_m = tl.program_id(0)
  # off_hz = tl.program_id(1)
  for start_m in range(16):
    for off_hz in range(32):
      print(f"start_m : {start_m}, off_hz : {off_hz}")
      off_z = off_hz // H
      off_h = off_hz % H
      print(f"off_z : {off_z},off_h : {off_h}")
      y_dim = Z * H * N_CTX
      print(f"y_dim : {y_dim}")
      offset_y = off_z * (N_CTX * H) + off_h * N_CTX
      print(f"offset_y : {offset_y}")
      qo_offset_y = offset_y + start_m * BLOCK_M
      print(f"qo_offset_y : {qo_offset_y}")
      # initialize offsets
      offs_m = start_m * BLOCK_M + torch.arange(0, BLOCK_M)
      offs_n = torch.arange(0, BLOCK_N)
      print(f"offs_m : {offs_m}")
      print(f"offs_n : {offs_n}")
      # initialize pointer to m and l
      m_i = torch.zeros([BLOCK_M], dtype=torch.float32) - float("inf")
      l_i = torch.zeros([BLOCK_M], dtype=torch.float32) + 1.0
      acc = torch.zeros([BLOCK_M, HEAD_DIM], dtype=torch.float32)
      # load scales
      qk_scale = sm_scale
      qk_scale *= 1.44269504  # 1/log(2)
      # q = desc_q.load([qo_offset_y, 0])
      print(f"q load : {[qo_offset_y, 0]}")
      # q = torch.randn([qo_offset_y, 0], dtype=dtype, device=DEVICE, requires_grad=True)
      # # print(f"q : {q}")
      if STAGE & 1:
                  _attn_fwd_inner(acc, l_i, m_i, q,  #
                                  desc_k, desc_v,  #
                                  offset_y, dtype, start_m, qk_scale,  #
                                  BLOCK_M, HEAD_DIM, BLOCK_N,  #
                                  4 - STAGE, offs_m, offs_n, N_CTX,  #
                                  warp_specialize)
      # stage 2: on-band
      if STAGE & 2:
                  _attn_fwd_inner(acc, l_i, m_i, q,  #
                                  desc_k, desc_v,  #
                                  offset_y, dtype, start_m, qk_scale,  #
                                  BLOCK_M, HEAD_DIM, BLOCK_N,  #
                                  2, offs_m, offs_n, N_CTX,  #
                                  warp_specialize)
      # epilogue
      # m_i += tl.math.log2(l_i)
      # acc = acc / l_i[:, None]
      # m_ptrs = M + off_hz * N_CTX + offs_m
      # tl.store(m_ptrs, m_i)
      # desc_o.store([qo_offset_y, 0], acc.to(dtype))
      if off_hz == 1:
          break
    if start_m == 1:
      print("------------------------------")
      return


def attention(q, k, v, causal, sm_scale, warp_specialize=True):
  # shape constraints
  HEAD_DIM_Q, HEAD_DIM_K = q.shape[-1], k.shape[-1]
  # when v is in float8_e5m2 it is transposed.
  HEAD_DIM_V = v.shape[-1]
  o = torch.empty_like(q)
  stage = 3 if causal else 1
  M = torch.empty((q.shape[0], q.shape[1], q.shape[2]), device=q.device, dtype=torch.float32)
  desc_q = q
  desc_v = v
  desc_k = k
  desc_o = o
  BLOCK_M = 64 #128
  BLOCK_N = 32 # 64 #128
  grid = (q.shape[2]// BLOCK_M, q.shape[0] * q.shape[1], 1)
  print(f"grid : {grid}")
  _attn_fwd(sm_scale, M, q.shape[0], q.shape[1],  #
            desc_q, desc_k, desc_v, desc_o,  #
            N_CTX=q.shape[2],  #
            HEAD_DIM=HEAD_DIM_K,  #
            BLOCK_M=BLOCK_M,  #
            BLOCK_N=BLOCK_N,  #
            FP8_OUTPUT=q.dtype == torch.float8_e5m2,  #
            STAGE=stage,  #
            warp_specialize=warp_specialize)

sm_scale = 0.5
causal = True
warp_specialize = False #True
BATCH = 4 #4
H = 8 #2
N_CTX = 1024 #1024
HEAD_DIM = 512 #64
dtype = torch.float16
q = torch.randn((BATCH, H, N_CTX, HEAD_DIM), dtype=dtype, device=DEVICE, requires_grad=True)
k = torch.randn((BATCH, H, N_CTX, HEAD_DIM), dtype=dtype, device=DEVICE, requires_grad=True)
v = torch.randn((BATCH, H, N_CTX, HEAD_DIM), dtype=dtype, device=DEVICE, requires_grad=True)

q = q.to(torch.float8_e5m2)
k = k.to(torch.float8_e5m2)
v = v.permute(0, 1, 3, 2).contiguous()
v = v.permute(0, 1, 3, 2)
v = v.to(torch.float8_e5m2)
print(f"V Shape : {v.shape}")

attention(q, k ,v, causal, sm_scale, warp_specialize)




V Shape : torch.Size([4, 8, 1024, 512])
grid : (16, 32, 1)
start_m : 0, off_hz : 0
off_z : 0,off_h : 0
y_dim : 32768
offset_y : 0
qo_offset_y : 0
offs_m : tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
        36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
        54, 55, 56, 57, 58, 59, 60, 61, 62, 63])
offs_n : tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31])
q load : [0, 0]
1
----inner-----
2
start_n : 0 : offset of k [0,0]
mask is used tensor([[ True, False, False,  ..., False, False, False],
        [ True,  True, False,  ..., False, False, False],
        [ True,  True,  True,  ..., False, False, False],
        ...,
        [ True,  True,  True,  ...,  True,  True,  True],
        [ True,  True,  True,  ...,  True,  True,  True],
        [ True,  Tr

## Backward

Here in backward we calulate the derivatives to finds the difference between the predicted value and the acutal value and we move towards the actual value by a learning rate. This is gradient descent in a nutshell.

### Derivatives

Lets setup the basic notations before to dig deep into the code. Here,
$$
O = PV
$$
Where P is
$$
P = softmax(QK^T)
$$
So if we break-down it into the derivatives, parital derivative of Value(V)
$$
\delta V = P^T\delta O
$$

parital derivative of P is
$$
\delta P = \delta OV^T
$$

We know partial derivative of softmax is,
$$
\delta S_{ij} = p_{ij} * (\delta P_{ij} - \sum_k \delta P_{ik} * p_{ik})
$$

Here we represent $\sum_k \delta P_{ik} p_{ik} = D_i$ which is represented as Delta in the code.
We have made use of O and $\delta O $ in our disposal by replacing $$\delta P * P = O \, \delta O$$

Since,
$$
\delta P = \delta O \, V^T \\
\delta P * P = \delta O \, V^T * P
$$

As we know $O = PV$ the above product is calulated as part of the pre-process.

### Function

As part of the backward process we need to calulate the difference between output(O) and the acutal value which gives us partial derivative if output ($\delta O$). First we make sure that strides of q, k, v, o, do are same then we create empty tensors for all the derivatives and rest of the variables. Important thing to note is we check that N_CTX is divisible by PRE_BLOCK or not.

This helps in defining the grid for _attn_bwd_proprocess properly and create delta. Here the grid defined is (N_CTX//PRE_BLOCK, BATCH_SIZE*HEAD). So as we can see we read PRE_BLOCK(BLOCK_M inside _attn_bwd_proprocess), N_CTX//PRE_BLOCK times and take whole HEAD_DIM and multiply O with $\delta O$ and sum it along the HEAD_DIM to get Delta as (BATCH_SIZE*HEAD*N_CTX,1) and its block as (BLOCK_M,).

For _attn_bwd we form the grid as (N_CTX//BLOCK_N1, 1, BATCH*N_HEAD). Here please dont get confused with 1 taken as a 2nd dimension as it is not used anywhere and just kept for extensibility since triton grids can be of 3 dimensions it seems to be added just for extensibility. Here we have divided the N_CTX into blocks of BLOCK_N1.

Inside the _attn_bwd we use the same trick of converting data into rows and columns using bhid % H & bhid // H and mutliplying them with the strides gives us the right offset show below
```
bhid = tl.program_id(2)
adj = (stride_h * (bhid % H) + stride_z * (bhid // H)).to(tl.int64)
```
We use these offset for all Q, K, V, DO, DQ, DV and since M(Maximum) and D(Delta) are alone y_dimension(BATCH_SIZE*HEAD*N_CTX) we create offset as
```
off_chz = (bhid * N_CTX).to(tl.int64)
```
Here we use BLK_SLICE_FACTOR to break BLOCK_M1 to more smaller blocks so that the masking is done efficently. This creates MASK_BLOCK_M1 as as we did in forward part we call here with Mask=True and Non Mask part with Mask=False. Hhere we used num_steps to diving the mask and the non-mask part. This is mask part
```
num_steps = BLOCK_N1 // MASK_BLOCK_M1
```
After mask is done we skip those by adding num_steps*MASK_BLOCK_M1 and getting offset added to start_m
```
start_m += num_steps * MASK_BLOCK_M1
num_steps = (N_CTX - start_m) // BLOCK_M1
```
which is defined intially for a BLOCK of N_CTX which is BLOCK_N1. Now lets talk about start_n which is used to create offset along N_CTX per HEAD per BATCH, so the value becomes
```
start_m = start_n
offs_n = start_n + tl.arange(0, BLOCK_N1)
```
So the k and v is defined with offsets as
```
k = tl.load(K + offs_n[:, None] * stride_tok + offs_k[None, :] * stride_d)
v = tl.load(V + offs_n[:, None] * stride_tok + offs_k[None, :] * stride_d)
```
Lets understand the _attn_bwd_dkdv function first. Here we don't create a triton grid based program but we create a simple triton funtion which executes on top if the grid elements with offsets provided to it. For this BLOCK_M1 is taken as MASK_BLOCK_M1 and BLOCK_N1 is same value just passed as a parameter from the pervious function. So the offsets become,
```
offs_m = start_m + tl.arange(0, BLOCK_M1)
offs_n = start_n + tl.arange(0, BLOCK_N1)
offs_k = tl.arange(0, HEAD_DIM)
```
We read q in a transposed way so that q can be directly applied for $kq^T = qk^T$.
```
qT_ptrs = Q + offs_m[None, :] * stride_tok + offs_k[:, None] * stride_d
```
Since we have already defined the offsets for k and given as an input to the function we don't have to break down the k anymore and use the value of k directly for calculation for $kq^T$
```
qkT = tl.dot(k, qT)
```
Now the same running softmax way is used but we have the maximum value already calculated from the forward function we use the same and calculate pT. We apply the mask on it when it is true and don't apply it when it is false and then we calulate $\delta V$. $\delta o$ is calculated using similar offset as qT but not in a transposed manner.
```
ppT = ppT.to(tl.float16)
dv += tl.dot(ppT, do)
```
During the calulations you will see some conversion from float32 to float16 and vice versa. This is done to mantain higher precision during calculation(float32) and then it is converted back to float16. It is a common pattern used in LLMs. Then we calulate patial derivative of softmax qk.
```
dpT = tl.dot(v, tl.trans(do)).to(tl.float32)
dsT = pT * (dpT - Di[None, :])
dsT = dsT.to(tl.float16)
```
Finally we calulate the $\delta k$ and increment the pointers for next iteration
```
dk += tl.dot(dsT, tl.trans(qT))
# Increment pointers.
curr_m += step_m
qT_ptrs += step_m * stride_tok
do_ptrs += step_m * stride_tok
```
Similar kind of offset is used in $\delta v$ & $\delta k$ pointers and stored back after the responses are obtained from the _attn_bwd_dkdv method.

Here we are using pid again and defining the start and end of the blocks for calulation of dq.
```
start_m = pid * BLOCK_M2
end_n = start_m + BLOCK_M2
```
However an important point to note is we should do assert on BLOCK_N1 = BLOCK_M2 as we are dividing the pid into BLOCK_N1 parts using (N_CTX//BLOCK_N1) so get the right offset in each tl program we need the above condition.
```
tl.static_assert(BLOCK_N1 == BLOCK_M2)
```
In this case now we use BLOCK_N2 for converting bigger BLOCKS into smaller BLOCKS using MASK_BLOCK_N2 during Mask=True. We use the same kind of technique here for calulating the num_steps.
```
num_steps = BLOCK_M2 // MASK_BLOCK_N2
```
Here before calling _attn_bwd_dq we define the start_n as
```
start_n = end_n - num_steps * MASK_BLOCK_N2
```
Important part to note is since we are only considering causal attention the stage 2 of _attn_bwd_dq will only run when all the 128x128 tokens are true. Which will happen only for old tokens in casual case. I found a bug in this case and suggested an improvement in the lower section.

Inside _attn_bwd_dq we load the blocks as BLOCK_M2 and BLOCK_N2 and calculate the qk and use the m obtained from the forward function and calculate the running softmax.
```
qk = tl.dot(q, kT)
p = tl.math.exp2(qk - m)
```
Then we calculate the $\delta p$, $\delta s$ and $\delta q$ using
```
dp = tl.dot(do, vT).to(tl.float32)
ds = p * (dp - Di[:, None])
ds = ds.to(tl.float16)
# Compute dQ.
# NOTE: We need to de-scale dq in the end, because kT was pre-scaled.
dq += tl.dot(ds, tl.trans(kT))
```

Finally we multiply $\delta q$ with $\ln(2)$ as we use exp2 for faster computation.
```
tl.math.exp2(qk - m)
```
But the derivate is not the same so we need to multiply it with $\ln(2)$ go get right result. If we don't do this then it will result in wrong values and the training might get affected.

### Example

To demostrate the offsets we have taken the triton code and converted all the lines into pytorch code. for tl programs we have used for loops based on the grid size. We take an example of shape torch.Size([4, 8, 1024, 512]) where 4 is the batch size, 8 is the no of heads, 1024 is context length and 512 is hidden dimensions per context token. For preprocessing we have take PRE_BLOCK as 128. and BLOCK sizes as
```
BLOCK_M1, BLOCK_N1, BLOCK_M2, BLOCK_N2 = 32, 128, 128, 32
```

#### _attn_bwd_preprocess

Here you can see the calculated grid comes as (8, 32) for _attn_bwd_preprocess. Since it is not triton we create 2 for loops to represent it inside the proprocess function. we run it for 2 iterations each so total of 4 tl programs represented using loops just to display the offsets for understanding. Here you can see as follows :

1. program_id(0) = 0 & program_id(1) = 0 : makes the offset of O and $\delta O$ run in shape 128x512. Since memory unit are contiguous so the last offset is (128x512)-512 = 65024 to (128x152)-1 = 65535. Since we are doing the sum along the hidden dimension Delta offsets lie from 0 to 127

2. program_id(0) = 0 & program_id(1) = 1 : Since we are increasing program_id(1) i.e. (BATCHxHEAD) by 1 so the offset move by 1024x512 = 524288. and runs same as above with last offset as 524288+(128x512)-512 = 589312 to 524288+(128x512)-1 = 589823. Similarly Delta moves by 1024 to become 1024 to 1151

3. program_id(0) = 1 & program_id(1) = 0 : Since we are inceasing the program_id(0) i.e. (N_CTX // PRE_BLOCK). The offset movement for 1st offset becomes 128x512 = 65536 to 128x512+512-1 = 66047 and similarly last offset becomes 65536+(128x512)-512=130560 to 65536+(128x512)-1=131071. Similarly Delta moves by 1024 to become 128 to 255 as we are looking into the next PRE_BLOCK(128) tokens

4. program_id(0) = 1 & program_id(1) = 1 : Since in this case we are incresing both ids by 1 so the 1st offset move becomes 1024x512+128x512 = 589824 to 1024x512+128x512+512-1 = 590335 and the last offset becomes 1024x512+128x512+128x512-512=654848 to 1024x512+128x512+128x512-1=655359. Similary Delta moves by 1024 + 128 = 1152 to 1279

#### _attn_bwd

Here you can see the calculated grid comes to Grid : (8, 1, 32) for _attn_bwd. I also printed the strides as these define the way the data sits in memory. These values are (4194304, 524288, 512, 1). We make use of these strides to move the offset appropriately. We use the same for loop kind of structure to identify the tl program ids. Here also we run the tl program for 2 iterations each per loop. This gives us the information for 4 tl programs, here are the details of the offset per tl program.

1. program_id(0) = 0 & program_id(1) = 0 : Since we are reading BLOCK_N1 size per tl program. So offsets of k and v give us the proper offset however we are adjusting the offset using adj variable which is 0 here which reads 1st set of 128 tokens of 1st context for 1st HEAD and 1st BATCH. which is 0 to 511 hidden layers per token so last offsets range from 128x512-512 = 65024 to 128x512-1 = 65535.

2. program_id(0) = 0 & program_id(1) = 1 :  Since we are moving the context by BLOCK_N1 the next set of BLOCK_N1 comes which is starts from 1st offset (128x512 = 65536 to 128x512+512-1 = 66047) and last offset (65536 + 128x512-512 = 130560 to 65536 + 128x512-1 = 131071).

3. program_id(0) = 1 & program_id(1) = 0 : In this case we are moving the offset by adj(524288) which is that batch and that head N_CTX and we add to the 1st point offsets to it.

4. program_id(0) = 1 & program_id(1) = 1 : Similarly for this adj(524288) which is that batch and that head N_CTX but next 128 tokens so we add to the 2nd point offsets to it.

So mulitple tl programs runs parallely to cover all the tokens per batch per head (program_id(0) adjusted by adj) and it is run in for BLOCK_N1 context (program_id(1) per 128 block size of whole N_CTX).

#### _attn_bwd_dkdv

Here we dont use the same tl program to call simple methods without any GRID. We call this method twice with MASK=True and MASK=False. For MASKED inputs we use the BULK_SLICE to form grid for a smaller N_CTX length in this case it is 16. Since we are not converting q and do to pointers and extracting chunks out of it. This is done by triton. We for better understanding use torch.randn to create exact chunks of data.

This same way of chunking done in k and v is now done for q and do. We also adjusted offset adj(Q and DO) however since we are slicing these we get (512,16) version of q and do for Mask=True and (512,32) version of q and do for Mask=True. Rest all is described above. However we break these chunks of 128 to n-steps of 32 and do all operations on top of it. We keep on moving the pointer by step_m*stride_tok. So matrix top product is done as k(32x512) * qT(512x16) = qkT(32x16) however this is 32x32 in case of mask=false. Rest of the math can be seen in the above.

Important thing to note is for Current Step 0 and Current Step 1 they lie beside each other so if you notice the mask for current step is True for all 1st offset and then from 2nd offset the false starts however for Current Step 1 all the first 3 offsets printed starts and ends with True.

Now we return back to the _attn_bwd where we start preparing the save dk and dv. Same kind of offset is used for dq and do.

#### _attn_bwd_dq

we run _attn_bwd_dq in BLOCKS of 16 N_CTX tokens at a time and due to this all the masks generated for 1 _attn_bwd_dq stand one beside another. As you can see from the output generated. The math remains the same however I found a bug while debugging offsets. When we consider causal attention  the BLOCKS which are Non-Masks areas were not even considered. Lets consider 128x128 blocks in N_CTX size of 1024. In the 1st iteration 128x128 will be a diagonal mask. Whereas in the next iteration there are 2 tiles from 128 to 256. 1st tile of 128x128 will be the non-mask part and next 128x128 will be the diagonal mask part. We are never considering the non-mask part of it. So I made a change by setting end_n with start_m in causal cases. This will call the _attn_bwd_dq for stage 2 and I also moved the offsets out of the _attn_bwd_dq for loop mask condition. Now
```
if MASK:
    offs_n = curr_n + tl.arange(0, BLOCK_N2)
    mask = (offs_m[:, None] >= offs_n[None, :])
    p = tl.where(mask, p, 0.0)
```
is replaced with 
```
offs_n = curr_n + tl.arange(0, BLOCK_N2)
if MASK:
    mask = (offs_m[:, None] >= offs_n[None, :])
    p = tl.where(mask, p, 0.0)
```
rest of the math remains same.

In [26]:
def _attn_bwd_preprocess(O, DO,  #
                         Delta,  #
                         Z, H, N_CTX,  #
                         BLOCK_M, HEAD_DIM  #
                         ):
  # Since PRE_BLOCK = BLOCK_M
  for pre_bloc_by_nctx in range(N_CTX // BLOCK_M):
    # BATCH = Z & N_HEAD = H
    for off_hz in range(Z * H):
      # If Dimension is 4x8x1024x512 then this is running 1024x512 where we are chunking them in 128x152 so pre-block = 8 and for each batch and each head so 8x4 = 32
      print(f"tl-program-ids : {pre_bloc_by_nctx} of {N_CTX // BLOCK_M}, {off_hz} of {Z * H}")
      off_m = pre_bloc_by_nctx * BLOCK_M + torch.arange(0, BLOCK_M)
      off_n = torch.arange(0, HEAD_DIM)
      # load
      offset_o_do = off_hz * HEAD_DIM * N_CTX + off_m[:, None] * HEAD_DIM + off_n[None, :]
      print(f"Offsets of o and do {offset_o_do.shape} : {offset_o_do}")
      # o = tl.load(O + off_hz * HEAD_DIM * N_CTX + off_m[:, None] * HEAD_DIM + off_n[None, :])
      # do = tl.load(DO + off_hz * HEAD_DIM * N_CTX + off_m[:, None] * HEAD_DIM + off_n[None, :]).to(tl.float32)
      # delta = tl.sum(o * do, axis=1)
      # write-back
      delta_offset = off_hz * N_CTX + off_m
      # delta is the sum of all hidden dimensions so its dimension is 4x8x1024 per 128 block of ctx
      print(f"Delta {delta_offset.shape} : {delta_offset}")
      # tl.store(Delta + off_hz * N_CTX + off_m, delta)
      if off_hz == 1:
        break
    if pre_bloc_by_nctx == 1:
        break


def _attn_bwd_dq(dq, q, K, V,  #
                 do, m, D,
                 # shared by Q/K/V/DO.
                 stride_tok, stride_d,  #
                 H, N_CTX,  #
                 BLOCK_M2,  #
                 BLOCK_N2,  #
                 HEAD_DIM,
                 # Filled in by the wrapper.
                 start_m, start_n, num_steps,  #
                 MASK):
    print(f"-------------_attn_bwd_dq--start_m--{start_m}--start_n--{start_n}----MASK-{MASK}-----")
    offs_m = start_m + torch.arange(0, BLOCK_M2)
    offs_n = start_n + torch.arange(0, BLOCK_N2)
    offs_k = torch.arange(0, HEAD_DIM)
    offset_kT = offs_n[:, None] * stride_tok + offs_k[None, :] * stride_d
    print(f"Offset of kT : {offset_kT.shape}")
    kT_ptrs = torch.randn((16,512))
    # kT_ptrs = K + offset_kT
    offset_vT = offs_n[None, :] * stride_tok + offs_k[:, None] * stride_d
    print(f"Offset of vT : {offset_vT.shape}")
    # vT_ptrs = V + offset_vT
    vT_ptrs = torch.randn((512,16))
    # D (= delta) is pre-divided by ds_scale.
    print(f"Di Offset : {offs_m.shape}")
    # Di = torch.load(D + offs_m)
    Di = torch.randn((128,))
    curr_n = start_n
    step_n = BLOCK_N2
    for blk_idx in range(num_steps):
        # kT = tl.load(kT_ptrs)
        # vT = tl.load(vT_ptrs)
        # kT = kT_ptrs
        # vT = vT_ptrs
        # qk = torch.dot(q, kT)
        # p = tl.math.exp2(qk - m)
        # Autoregressive masking.
        offs_n = curr_n + torch.arange(0, BLOCK_N2)
        print(f"Offset of n ({offs_n.shape}): {offs_n}")
        print(f"Offset of m ({offs_m.shape}) : {offs_m}")
        if MASK:
            mask = (offs_m[:, None] >= offs_n[None, :])
            print(f"Mask ({mask.shape}) : {mask}")
            # p = torch.where(mask, p, 0.0)
        # Compute dP and dS.
        # dp = tl.dot(do, vT).to(tl.float32)
        # ds = p * (dp - Di[:, None])
        # ds = ds.to(tl.float16)
        # Compute dQ.
        # NOTE: We need to de-scale dq in the end, because kT was pre-scaled.
        # dq += tl.dot(ds, tl.trans(kT))
        # Increment pointers.
        curr_n += step_n
        # kT_ptrs += step_n * stride_tok
        # vT_ptrs += step_n * stride_tok
    # return dq
    print(f"-------------end of_attn_bwd_dq-------")

def _attn_bwd_dkdv(dk, dv,  #
                   Q, k, v, sm_scale,  #
                   DO,  #
                   M, D,  #
                   # shared by Q/K/V/DO.
                   stride_tok, stride_d,  #
                   H, N_CTX, BLOCK_M1,  #
                   BLOCK_N1,  #
                   HEAD_DIM,  #
                   # Filled in by the wrapper.
                   start_n, start_m, num_steps,  #
                   MASK):
    print(f"--------_attn_bwd_dkdv-(MASK : {MASK})-------------")
    offs_m = start_m + torch.arange(0, BLOCK_M1)
    offs_n = start_n + torch.arange(0, BLOCK_N1)
    offs_k = torch.arange(0, HEAD_DIM)
    qT_offset = offs_m[None, :] * stride_tok + offs_k[:, None] * stride_d
    do_offset = offs_m[:, None] * stride_tok + offs_k[None, :] * stride_d

    # print(f"({start_m}, {start_n}) : qT_offset({qT_offset.shape}) : {qT_offset} : do_offset({do_offset.shape}) : {do_offset}")
    qT_ptrs = torch.randn((512,16)) if MASK else torch.randn((512,32))
    do_ptrs = torch.randn((512,16)) if MASK else torch.randn((512,32))
    # # BLOCK_N1 must be a multiple of BLOCK_M1, otherwise the code wouldn't work.
    # # tl.static_assert(BLOCK_N1 % BLOCK_M1 == 0)
    curr_m = start_m
    step_m = BLOCK_M1
    for blk_idx in range(num_steps):
    #     # qT = tl.load(qT_ptrs)
          # qT = qT_ptrs
    #     # Load m before computing qk to reduce pipeline stall.
          offs_m = curr_m + torch.arange(0, BLOCK_M1)
          print(f"Current Step : {blk_idx} : offs_m({offs_m.shape}) : {offs_m}")
    #     # m = tl.load(M + offs_m)
          # m = torch.randn(((32,)))
    #     qkT = tl.dot(k, qT)
    #     pT = tl.math.exp2(qkT - m[None, :])
    #     # Autoregressive masking.
          if MASK:
              mask = (offs_m[None, :] >= offs_n[:, None])
              print(f"Mask ({mask.shape}) : {mask}")
    #         pT = tl.where(mask, pT, 0.0)
    #     do = tl.load(do_ptrs)
    #     # Compute dV.
    #     ppT = pT
    #     ppT = ppT.to(tl.float16)
    #     dv += tl.dot(ppT, do)
    #     # D (= delta) is pre-divided by ds_scale.
    #     Di = tl.load(D + offs_m)
    #     # Compute dP and dS.
    #     dpT = tl.dot(v, tl.trans(do)).to(tl.float32)
    #     dsT = pT * (dpT - Di[None, :])
    #     dsT = dsT.to(tl.float16)
    #     dk += tl.dot(dsT, tl.trans(qT))
    #     # Increment pointers.
          curr_m += step_m
        # qT_ptrs += step_m * stride_tok
    #     do_ptrs += step_m * stride_tok
          if blk_idx == 1:
              break
    # return dk, dv

def _attn_bwd(Q, K, V, sm_scale,  #
              DO,  #
              DQ, DK, DV,  #
              M, D,
              # shared by Q/K/V/DO.
              stride_z, stride_h, stride_tok, stride_d,  #
              H, N_CTX,  #
              causal,
              BLOCK_M1,  #
              BLOCK_N1,  #
              BLOCK_M2,  #
              BLOCK_N2,  #
              BLK_SLICE_FACTOR,  #
              HEAD_DIM):
    LN2 = 0.6931471824645996  # = ln(2)
    # bhid = tl.program_id(2)
    for bhid in range(32):
    # pid = tl.program_id(0)
      for pid in range(8):
        off_chz = (bhid * N_CTX)
        row = bhid // H
        col = bhid % H
        # adj will move along per BxH
        adj = (stride_h * col + stride_z * row)
        print(f"(bhid,pid to 32,8) : ({bhid},{pid}) : Row : {row}, Col : {col}, off_chz : {off_chz} adj : {adj}")
        offs_k = torch.arange(0, HEAD_DIM)
        start_n = pid * BLOCK_N1
        start_m = start_n
        MASK_BLOCK_M1 = BLOCK_M1 // BLK_SLICE_FACTOR
        # print(f"MASK_BLOCK_M1 : {MASK_BLOCK_M1}")
        offs_n = start_n + torch.arange(0, BLOCK_N1)

        dv = torch.zeros([BLOCK_N1, HEAD_DIM], dtype=torch.float32)
        dk = torch.zeros([BLOCK_N1, HEAD_DIM], dtype=torch.float32)

        # load K and V: they stay in SRAM throughout the inner loop.
        # k = tl.load(K + offs_n[:, None] * stride_tok + offs_k[None, :] * stride_d)
        # v = tl.load(V + offs_n[:, None] * stride_tok + offs_k[None, :] * stride_d)
        offset_k_v = offs_n[:, None] * stride_tok + offs_k[None, :] * stride_d
        print(f"Offsets({offset_k_v.shape}) of k or v {offset_k_v}")
        num_steps = BLOCK_N1 // MASK_BLOCK_M1
        print(f"MASK : True : start_m : {start_m} : Num Steps : {num_steps}")
        # Mask is true
        _attn_bwd_dkdv(dk, dv,  #
                            Q, k, v, sm_scale,  #
                            DO,  #
                            M, D,  #
                            stride_tok, stride_d,  #
                            H, N_CTX,  #
                            MASK_BLOCK_M1, BLOCK_N1, HEAD_DIM,  #
                            start_n, start_m, num_steps,  #
                            MASK=True  #
                            )
        start_m += num_steps * MASK_BLOCK_M1
        num_steps = (N_CTX - start_m) // BLOCK_M1
        print(f"MASK : False : start_m : {start_m} : Num Steps : {num_steps}")

        # # Compute dK and dV for non-masked blocks.
        # # Mask is false
        _attn_bwd_dkdv(  #
            dk, dv,  #
            Q, k, v, sm_scale,  #
            DO,  #
            M, D,  #
            stride_tok, stride_d,  #
            H, N_CTX,  #
            BLOCK_M1, BLOCK_N1, HEAD_DIM,  #
            start_n, start_m, num_steps,  #
            MASK=False  #
        )
        print("-------End of _attn_bwd_dkdv-----------")
        dq_offset = offs_n[:, None] * stride_tok + offs_k[None, :] * stride_d
        print(f"Offset of DV : {dq_offset}")
        # dv_ptrs = DV + dq_offset
        # tl.store(dv_ptrs, dv)

        # # Write back dK.
        # dk *= sm_scale
        dk_offset = offs_n[:, None] * stride_tok + offs_k[None, :] * stride_d
        print(f"Offset of DK : {dk_offset}")
        # # dk_ptrs = DK + dk_offset
        # # tl.store(dk_ptrs, dk)

        # # THIS BLOCK DOES DQ:
        start_m = pid * BLOCK_M2
        end_n = start_m + BLOCK_M2

        MASK_BLOCK_N2 = BLOCK_N2 // BLK_SLICE_FACTOR
        offs_m = start_m + torch.arange(0, BLOCK_M2)
        print(f"Offset of m ({offs_m.shape}) : {offs_m}")
        offs_q = offs_m[:, None] * stride_tok + offs_k[None, :] * stride_d
        print(f"Offset of q ({offs_q.shape}) : {offs_q}")
        # q = tl.load(Q + offs_q)
        dq = torch.zeros([BLOCK_M2, HEAD_DIM], dtype=torch.float32)
        offs_do = offs_m[:, None] * stride_tok + offs_k[None, :] * stride_d
        print(f"Offset of do ({offs_do.shape}) : {offs_do}")
        # do = tl.load(DO + offs_do)
        # # m = tl.load(M + offs_m)
        m = torch.randn((128,))
        m = m[:, None]

        # # Compute dQ for masked (diagonal) blocks.
        # # NOTE: This code scans each row of QK^T backward (from right to left,
        # # but inside each call to _attn_bwd_dq, from left to right), but that's
        # # not due to anything important.  I just wanted to reuse the loop
        # # structure for dK & dV above as much as possible.
        num_steps = BLOCK_M2 // MASK_BLOCK_N2
        print(f"({bhid},{pid}) MASK : True : start_m : {start_m} start_n : {end_n - num_steps * MASK_BLOCK_N2} : Num Steps : {num_steps}")
        _attn_bwd_dq(dq, q, K, V,  #
                          do, m, D,  #
                          stride_tok, stride_d,  #
                          H, N_CTX,  #
                          BLOCK_M2, MASK_BLOCK_N2, HEAD_DIM,  #
                          start_m, end_n - num_steps * MASK_BLOCK_N2, num_steps,  #
                          MASK=True  #
                          )

        if causal:
          end_n = start_m
        else:
          end_n -= num_steps * MASK_BLOCK_N2

        print(f"end_n : {end_n}")
        # # stage 2
        num_steps = end_n // BLOCK_N2
        print(f"({bhid},{pid}) MASK : False : start_m : {start_m} start_n : {end_n - num_steps * BLOCK_N2} : Num Steps : {num_steps}")
        _attn_bwd_dq(dq, q, K, V,  #
                          do, m, D,  #
                          stride_tok, stride_d,  #
                          H, N_CTX,  #
                          BLOCK_M2, BLOCK_N2, HEAD_DIM,  #
                          start_m, end_n - num_steps * BLOCK_N2, num_steps,  #
                          MASK=False  #
                          )
        # Write back dQ.
        # dq_offset = offs_m[:, None] * stride_tok + offs_k[None, :] * stride_d
        # print(f"Offset of DQ : {dq_offset}")
        # dq_ptrs = DQ + dq_offset
        # Here while using softmax we used 2^x instead of e^x so when we do a derivative of it we need to mulitply it with (ln 2)
        # as derivative of e^x is e^x but derivative of 2^x is (ln 2).2^x
        # as dQ = d(qk)K = dS K so dS propotional to (ln 2)2^(qk)
        # dq *= LN2
        # tl.store(dq_ptrs, dq)
        print("-------End of _attn_bwd-----------")
        if pid == 2:
          break
      if bhid == 2:
        return


sm_scale = 0.5
causal = True
warp_specialize = False #True
BATCH = 4 #4
H = 8 #2
N_CTX = 1024 #1024
HEAD_DIM = 512#32 #64
dtype = torch.float16
q = torch.randn((BATCH, H, N_CTX, HEAD_DIM), dtype=dtype, device=DEVICE, requires_grad=True)
k = torch.randn((BATCH, H, N_CTX, HEAD_DIM), dtype=dtype, device=DEVICE, requires_grad=True)
v = torch.randn((BATCH, H, N_CTX, HEAD_DIM), dtype=dtype, device=DEVICE, requires_grad=True)

# q = q.to(torch.float8_e5m2)
# k = k.to(torch.float8_e5m2)
v = v.permute(0, 1, 3, 2).contiguous()
v = v.permute(0, 1, 3, 2)
# v = v.to(torch.float8_e5m2)
print(f"V Shape : {v.shape}")
o = torch.randn_like(q)
do = torch.randn_like(o)
dq = torch.empty_like(q)
dk = torch.empty_like(k)
dv = torch.empty_like(v)
BATCH, N_HEAD, N_CTX = q.shape[:3]
PRE_BLOCK = 128
NUM_WARPS, NUM_STAGES = 4, 5
BLOCK_M1, BLOCK_N1, BLOCK_M2, BLOCK_N2 = 32, 128, 128, 32
BLK_SLICE_FACTOR = 2
RCP_LN2 = 1.4426950408889634  # = 1.0 / ln(2)
arg_k = k
arg_k = arg_k * (sm_scale * RCP_LN2)
PRE_BLOCK = 128
pre_grid = (N_CTX // PRE_BLOCK, BATCH * N_HEAD)
print(f"Pre Grid : {pre_grid}")

M = torch.randn((q.shape[0], q.shape[1], q.shape[2]), device=q.device, dtype=torch.float32)
delta = torch.empty_like(M)
print("-------Start of _attn_bwd_preprocess-----------")
_attn_bwd_preprocess(
            o, do,  #
            delta,  #
            BATCH, N_HEAD, N_CTX,  #
            BLOCK_M=PRE_BLOCK, HEAD_DIM=HEAD_DIM  #
        )
print("-------End of _attn_bwd_preprocess-----------")
grid = (N_CTX // BLOCK_N1, 1, BATCH * N_HEAD)
print(f"Grid (N_CTX // BLOCK_N1, 1, BATCH * N_HEAD): {grid}")
print(f"Strides of q : {q.stride(0)}, {q.stride(1)}, {q.stride(2)}, {q.stride(3)}")
print("-------Start of _attn_bwd-----------")
_attn_bwd(
            q, arg_k, v, sm_scale, do, dq, dk, dv,  #
            M, delta,  #
            q.stride(0), q.stride(1), q.stride(2), q.stride(3),  #
            N_HEAD, N_CTX,  #
            causal,
            BLOCK_M1=BLOCK_M1, BLOCK_N1=BLOCK_N1,  #
            BLOCK_M2=BLOCK_M2, BLOCK_N2=BLOCK_N2,  #
            BLK_SLICE_FACTOR=BLK_SLICE_FACTOR,  #
            HEAD_DIM=HEAD_DIM,  #
            # num_warps=NUM_WARPS,  #
            # num_stages=NUM_STAGES  #
        )

V Shape : torch.Size([4, 8, 1024, 512])
Pre Grid : (8, 32)
-------Start of _attn_bwd_preprocess-----------
tl-program-ids : 0 of 8, 0 of 32
Offsets of o and do torch.Size([128, 512]) : tensor([[    0,     1,     2,  ...,   509,   510,   511],
        [  512,   513,   514,  ...,  1021,  1022,  1023],
        [ 1024,  1025,  1026,  ...,  1533,  1534,  1535],
        ...,
        [64000, 64001, 64002,  ..., 64509, 64510, 64511],
        [64512, 64513, 64514,  ..., 65021, 65022, 65023],
        [65024, 65025, 65026,  ..., 65533, 65534, 65535]])
Delta torch.Size([128]) : tensor([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
         14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,
         28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,  41,
         42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,  55,
         56,  57,  58,  59,  60,  61,  62,  63,  64,  65,  66,  67,  68,  69,
         70,  71,  72,  73,  74,  75,

## Error in Triton Manual

While calculating the num_steps for MASK and NON_MASK cases for $\delta q$ calculation we were not considering the areas which have mask=false like in case where if N_CTX = 1024 and for pid=1 where we consider range of N_CTX from 128 to 256. We have 2 tiles here of 128x128. 1st tile is the Non-mask tile as all values need to be considered and 2nd tile is the Mask tile where we form the diagonal mask.
change suggested is adding causal as a parameter to the _attn_bwd kernel and checking its value while executing the stage 2 of the _attn_bwd_dq function.

```
if causal:
  end_n = start_m
else:
  end_n -= num_steps * MASK_BLOCK_N2
```
and in 
This solves the problem of Masked and Non-Masked areas in a casual attention. We also need to replace
 ```
if MASK:
    offs_n = curr_n + tl.arange(0, BLOCK_N2)
    mask = (offs_m[:, None] >= offs_n[None, :])
    p = tl.where(mask, p, 0.0)
```
with

```
offs_n = curr_n + tl.arange(0, BLOCK_N2)
if MASK:
    mask = (offs_m[:, None] >= offs_n[None, :])
    p = tl.where(mask, p, 0.0)
```
in the _attn_bwd_dq

### Suggestion  

However an important point to note is we should do assert on BLOCK_N1 = BLOCK_M2 as we are dividing the pid into BLOCK_N1 parts using (N_CTX//BLOCK_N1) so get the right offset in each tl program we need the above condition.
```
tl.static_assert(BLOCK_N1 == BLOCK_M2)
```

## Out of scope

Triton is a way to do matrix operations efficiently using smaller size BLOCK of memory. However this has a limitation of running on a single GPU. To run it across GPUs in a distributed enviroment. Every type of GPUs have there own way of handling GPU instructions like CUDA has NCCL, AMD has RCCL. So we need to cordinate between the GPUs to apply those instructions. There are multiple ways of communication. Most commonly used in this case is 2 way pairing. However due to this all GPUs must enter the same operation. This wastes a lot of time as it has to sync accross all GPUs.

So NVSHMEM was introduced which supports 1 sided communication and to any GPU at any time which leads to Partitioned Global Address Space across multiple GPUs.

However there are a lot of libraries written on top of these to make it easier to use. However it takes away the flexibilty of operating GPUs in a custom way.

## Conclusion

Triton is a single GPU optimised way to do matrix operation by breaking down matrices into smaller blocks. So there is a scaling limit to it. To cross this limit we need to make used of Distributed GPUs and Partitioned Global Address Space(PGAS) creation to achieve better results. However some parallelism can be acheived by pipeling the components but it will also limit the execution of a single node in a pipeline to a single GPU. 

# Reference

Triton Manual : https://triton-lang.org/main/index.html

ChatGPT when i couldn't find any reason for some part fo code.
